In [1]:
import os
import pandas as pd
from openai import OpenAI

import numpy as np
import json

In [2]:
from dotenv import load_dotenv
load_dotenv()

# Access the environment variables from the .env file
api_key = os.environ.get('API_KEY')
client = OpenAI(api_key=api_key)

In [3]:
repo_dir = "/Users/haya1/Documents/LanguageModel_Labels/congressional_bills"
os.chdir(repo_dir)

data_dir = os.path.join(repo_dir, "01_cleaning_and_data")
bills = pd.read_csv(os.path.join(data_dir, "bills_no_prompts.csv")).head(10) # TODO: remove .head(10)
print(f"Loaded bills data, n = {len(bills)}")

Loaded bills data, n = 10


In [4]:
prompts_dir = os.path.join(repo_dir, "02_prompting")
base_prompt = open(os.path.join(prompts_dir, "base_prompt_json.txt"), 'r').read()
print("Loaded base question")
print(base_prompt)

Loaded base question
%sHere is a description of a bill introduced in the U.S. Congress:
"%s."

Please classify this description into one of the following categories:
1. Macroeconomics
2. Civil Rights, Minority Issues, and Civil Liberties
3. Health
4. Agriculture
5. Labor and Employment
6. Education
7. Environment
8. Energy
9. Immigration
10. Transportation
11. Law, Crime, and Family Issues
12. Social Welfare
13. Community Development and Housing Issues
14. Banking, Finance, and Domestic Commerce
15. Defense
16. Space, Science, Technology, and Communications
17. Foreign Trade
18. International Affairs and Foreign Aid
19. Government Operations
20. Public Lands and Water Management
21. Arts and Entertainment

%s

Output a JSON object structured like: {
    “category”: an integer from 1 to 21 of the bill category,
    “confidence”: a float number from 0 to 1 with two decimal places of your confidence in your bill classification%s
}


In [5]:
prompting_strategies = pd.DataFrame(data={
    "Name": [
        "none", 
        "persona", "persona", "persona", "persona", 
        "cot", "cot", "cot", 
        "fewshot", "fewshot", "fewshot"
        ],
    "BeforeQuestion": [
        "",
        "You are a knowledgeable political analyst. ",
        "Answer this question as if you are a political scientist that studies legislation in the United States Congress. ",
        "Answer this question as if you are an expert in United States politics. ",
        "Answer this question as if you were a helpful research assistant for a political scientist. ",
        "", "", "",
        "", "", "",
        ],
    "BeforeAnswer": [
        "", 
        "", "", "", "", 
        "Think carefully.", 
        "Let's think step by step. Lay out each step.", 
        "Please provide an explanation for your answer.",
        "", "", "",
    ],
    "Explanation":[
        "", 
        "", "", "", "",
        ',\n\t"explanation": a one-sentence explanation of your bill category answer',
        ',\n\t"explanation": a one-sentence explanation of your bill category answer',
        ',\n\t"explanation": a one-sentence explanation of your bill category answer',
        "", "", "",
    ]
})

In [6]:
MODEL = "gpt-3.5-turbo"
TEMPERATURE = 1
NUM_RESPONSES = 1
# SYSTEM_PROMPT = ""

# define a response function that gives us the LLM's response to a user prompt
def classify_llm(prompt):
    llm_out = client.chat.completions.create(
        model = MODEL,
        logprobs = True,
        n = NUM_RESPONSES,
        temperature = TEMPERATURE,
        response_format={ "type": "json_object" },
        messages=[
            # {"role": "system",  #  role (either "system", "user", or "assistant") The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
            #  "content": system},
            {"role": "user", 
            "content": prompt},
            # {"role": "assistant",
            #  "content": examples} # Assistant messages store previous assistant responses, but can also be written by you to give examples of desired behavior.
        ]
    ).choices[0].message.content

    llm_out_json = json.loads(llm_out)

    out = {"category": np.nan, "confidence": np.nan, "explanation": np.nan}

    if "category" in llm_out_json.keys():
        out["category"] = llm_out_json["category"]

    if "confidence" in llm_out_json.keys():
        out["confidence"] = llm_out_json["confidence"]

    if "explanation" in llm_out_json.keys():
        out["explanation"] = llm_out_json["explanation"]
    
    return out

In [7]:
# names = bills["company_name"]
# headlines = bills["headline"]
bill_ids = []
majors = []
major_texts = []
descriptions = []
prompts = []
responses = []

llm_major = []
llm_confidence = []
llm_explanation = []

for index, bill in bills.iterrows():
    bill_id = bill["BillID_corrected"]
    major = bill["Major_recoded"]
    major_text = bill["MajorText"]
    description = bill["Description"]
    
    for _, strategy in prompting_strategies.iterrows():
        bill_ids.append(bill_id)
        majors.append(major)
        major_texts.append(major_text)
        descriptions.append(description)

        prompt = base_prompt % (strategy["BeforeQuestion"], description, strategy["BeforeAnswer"], strategy["Explanation"])
        prompts.append(prompt)

        llm_answer = classify_llm(prompt)
        llm_major.append(llm_answer["category"])
        llm_confidence.append(llm_answer["confidence"])
        llm_explanation.append(llm_answer["explanation"])


prompts_and_responses = pd.DataFrame(data = {
    "BillID": bill_ids,
    "Major": majors,
    "MajorText": major_texts,
    "Description": descriptions,
    "Prompt": prompts,
    "MajorLLM": llm_major,
    "ConfidenceLLM": llm_confidence,
    "ExplanationLLM": llm_explanation
})

prompts_and_responses.to_csv(os.path.join(prompts_dir, "prompts_and_responses.csv"), index=False)